In [ ]:
import sys
!{sys.executable} -m pip install gurobipy
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

df = pd.read_csv('Promotions.csv', index_col=0)
categories = df.index.tolist()
staff = df.columns.tolist()
q = df.stack().to_dict()

df2 = pd.read_csv("Preferences.csv", index_col=0)
trips = df2.index.tolist()
r = df2.stack().to_dict()


df3 = pd.read_csv("TripCategories.csv", index_col=0)
c = df3.stack().to_dict()

df4= pd.read_csv("TripDays.csv", index_col=0)
days= df4.columns.tolist()
d = df4.stack().to_dict()

df5= pd.read_csv("Weighted.csv", index_col=0)
e = df5.stack().to_dict()
Wij = {
    (i,j): sum(c[i,k] * e.get((k,j), 0) for k in categories)
    for i in trips for j in staff
}
#Modeling
m = gp.Model("Scheduling")

#Decision Variables
x = m.addVars(trips, staff, vtype=GRB.BINARY, name="x")
y = m.addVars(trips, staff, vtype=GRB.BINARY, name ="y")
unschedulablex = m.addVars(trips, vtype=GRB.BINARY, name="unschedulable")
unschedulabley = m.addVars(trips, vtype=GRB.BINARY, name="unschedulable")
extra_overnights = m.addVars(staff, vtype=GRB.INTEGER, name="extra_overnights")

#Constraints
m.addConstrs(sum(x[i,j] for j in staff) + unschedulablex[i] == 1 for i in trips)
m.addConstrs(sum(y[i,j] for j in staff) + unschedulabley[i] == 1 for i in trips)
m.addConstrs(r[i,j] >= x[i,j] for i in trips for j in staff)
m.addConstrs(r[i,j] >= y[i,j] for i in trips for j in staff)
m.addConstrs(sum(c[i,"Overnight "]*x[i,j] for i in trips) - extra_overnights[j] <=1 for j in staff)
m.addConstrs(sum(c[i,"Overnight "]*y[i,j] for i in trips) - extra_overnights[j] <=1 for j in staff)
#m.addConstrs(extra_overnights[j] <= 1 for j in staff)
m.addConstrs(sum(q[k,j]*c[i,k] for k in categories) >= x[i,j] for i in trips for j in staff)
m.addConstrs(sum((1-q[k,j])*c[i,k] for k in categories) >= y[i,j] for i in trips for j in staff)
m.addConstrs(sum(d[i,l]*x[i,j] for i in trips) + sum(d[i,l]*y[i,j] for i in trips) <= 1 for l in days for j in staff)
m.addConstrs((2*(len(trips))/len(staff))+1 -sum(x[i,j] for i in trips) - sum(y[i,j] for i in trips) >= 0 for j in staff)
m.addConstrs((2*(len(trips))/len(staff))-1 -sum(x[i,j] for i in trips) - sum(y[i,j] for i in trips) <= 0 for j in staff)
m.addConstr(sum(unschedulablex[i] for i in trips) <= 3)
m.addConstr(sum(unschedulabley[i] for i in trips) <= 0)

#Objective Function
m.setObjective(
    sum((r[i,j] / max(Wij[i,j], 1e-6)) * x[i,j] for i in trips for j in staff) +
    sum((r[i,j] / max(Wij[i,j], 1e-6)) * y[i,j] for i in trips for j in staff) +
    sum(10000 * unschedulablex[i] for i in trips) +
    sum(10000 * unschedulabley[i] for i in trips) +
    sum(20*(extra_overnights[j])**2 for j in staff),
    GRB.MINIMIZE
)
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"{'Trip':<15} | {'Lead Guide':<15} (Rank) | {'Asst Guide':<15} (Rank)")
    print("-" * 65)

    for i in trips:
        lead_name = "NONE"
        lead_rank = 0
        asst_name = "NONE"
        asst_rank = 0
       
   
        for j in staff:
            if x[i,j].X > 0.5:
                lead_name = j
                lead_rank = r[i,j]
                break
       
   
        for j in staff:
            if y[i,j].X > 0.5:
                asst_name = j
                asst_rank = r[i,j]
                break
        print(f"{i:<15} | {lead_name:<15} ({lead_rank:>2}) | {asst_name:<15} ({asst_rank:>2})")

    total_rank_sum = sum(r[i,j] * x[i,j].X for i in trips for j in staff) + \
                     sum(r[i,j] * y[i,j].X for i in trips for j in staff)
   
    print("-" * 30)
    print(f"Total Rank Sum (Satisfaction): {int(total_rank_sum)}")
    print("-" * 30)

    results_list = []

    for i in trips:
        row = {
            "Trip": i,
            "Lead": "N/A",
            "Lead_Rank": 0,
            "Assistant": "N/A",
            "Assistant_Rank": 0
        }
        for j in staff:
            if x[i,j].X > 0.5:
                row["Lead"] = j
                row["Lead_Rank"] = r[i,j]
                break
        for j in staff:
            if y[i,j].X > 0.5:
                row["Assistant"] = j
                row["Assistant_Rank"] = r[i,j]
                break
       
        results_list.append(row)

    results_df = pd.DataFrame(results_list)
    results_df.to_csv("Final_Schedule.csv", index=False)

    staff_summary = []

    print(f"{'Staff Member':<15} | {'Assigned Trips':<40} | {'Total Rank'}")
    print("-" * 75)

    for j in staff:
        assigned_trips = []
        personal_rank_sum = 0
       
        for i in trips:
            if x[i,j].X > 0.5:
                assigned_trips.append(f"{i} (Lead)")
                personal_rank_sum += r[i,j]
       

        for i in trips:
            if y[i,j].X > 0.5:
                assigned_trips.append(f"{i} (Asst)")
                personal_rank_sum += r[i,j]
       
        trips_str = ", ".join(assigned_trips) if assigned_trips else "None"
       
        print(f"{j:<15} | {trips_str:<40} | {personal_rank_sum}")

        staff_summary.append({
            "Staff Member": j,
            "Trips": trips_str,
            "Total Individual Rank": personal_rank_sum
        })

    pd.DataFrame(staff_summary).to_csv("Staff_Assignments_Summary.csv", index=False)

else:
    print("No optimal solution found.")